In [1]:
import pickle as pkl

import numpy as np
from tqdm import tqdm

#### Objective

Using computed similarities build a sparse matrix with the distances (in trigram similarity space) between all pairs of vendors.

#### Depends on:

In [ ]:
vendor_to_idx_map_fname = "./data/norm_vendors_list_to_idx_map.pkl"
vendors_similarities_output_fname = "./data/vendors_similarities_trigrams_only.pkl"

#### Generates:

In [ ]:
norm_vendors_csr_matrix_fname = "./data/norm_vendors_csr_matrix_trigrams_only.pkl"

------------------------

In [2]:
with open(vendor_to_idx_map_fname, "rb") as f:
    norm_vendors_str = pkl.load(f)

In [3]:
norm_vendors_str[:5]

['AAA GLASS MIRROR',
 'AAA MAILBOX SALES SERVI',
 'AAA OKLAHOMA',
 'AAA OKLAHOMA MBR DUES',
 'AAA STRIPING SEALING']

In [6]:
with open(vendors_similarities_output_fname, "rb") as f:
    similarities = pkl.load(f)

In [7]:
len(similarities), len(vendors_idx_to_name), len(vendors_name_to_idx)

(30624, 30624, 30624)

In [8]:
import scipy

In [9]:
sims_array = scipy.sparse.lil_matrix(
    (len(similarities), len(similarities)), dtype=float
)

In [10]:
sims_array

<30624x30624 sparse matrix of type '<class 'numpy.float64'>'
	with 0 stored elements in List of Lists format>

In [11]:
similarities[0]

{'vendor_name': 'AAA GLASS MIRROR',
 'others': [{'vendor_name': 'ATLAS GLASS MIRROR', 'sim': 0.7857142857142857},
  {'vendor_name': 'CENTRAL GLASS MIRROR', 'sim': 0.7857142857142857},
  {'vendor_name': 'CLOUSE GLASS MIRROR', 'sim': 0.7857142857142857},
  {'vendor_name': 'DYER GLASS AND MIRROR', 'sim': 0.7142857142857143},
  {'vendor_name': 'FOX GLASS MIRROR', 'sim': 0.7857142857142857},
  {'vendor_name': 'GLASS MIRROR', 'sim': 0.7142857142857143}]}

In [12]:
for vendor_1 in tqdm(similarities):
    source_idx = vendors_name_to_idx[vendor_1["vendor_name"]]
    for vendor_2 in vendor_1["others"]:
        target_idx = vendors_name_to_idx[vendor_2["vendor_name"]]
        sims_array[source_idx, target_idx] = vendor_2["sim"]

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 30624/30624 [00:00<00:00, 44556.02it/s]


In [13]:
sims_array

<30624x30624 sparse matrix of type '<class 'numpy.float64'>'
	with 566265 stored elements in List of Lists format>

In [14]:
del similarities  # about 7 Gigs

In [15]:
csr_matrix = scipy.sparse.csr_array(sims_array)

In [16]:
csr_matrix

<30624x30624 sparse array of type '<class 'numpy.float64'>'
	with 566265 stored elements in Compressed Sparse Row format>

In [17]:
with open(norm_vendors_csr_matrix_fname, "wb") as f:
    pkl.dump(
        csr_matrix,
        f,
    )